In [2]:
import pandas as pd

files = [
    "UpdatedResumeDataSet.csv",
    "resume_data_for_ranking.csv",
    "all_job_post.csv"
]

for file in files:
    df = pd.read_csv(file)
    print("\n" + "="*80)
    print(file)
    print(df.shape)
    print(df.columns.tolist())
    print(df.head(2))


UpdatedResumeDataSet.csv
(962, 2)
['Category', 'Resume']
       Category                                             Resume
0  Data Science  Skills * Programming Languages: Python (pandas...
1  Data Science  Education Details \r\nMay 2013 to May 2017 B.E...

resume_data_for_ranking.csv
(9544, 35)
['address', 'career_objective', 'skills', 'educational_institution_name', 'degree_names', 'passing_years', 'educational_results', 'result_types', 'major_field_of_studies', 'professional_company_names', 'company_urls', 'start_dates', 'end_dates', 'related_skils_in_job', 'positions', 'locations', 'responsibilities', 'extra_curricular_activity_types', 'extra_curricular_organization_names', 'extra_curricular_organization_links', 'role_positions', 'languages', 'proficiency_levels', 'certification_providers', 'certification_skills', 'online_links', 'issue_dates', 'expiry_dates', 'job_position_name', 'educationaL_requirements', 'experiencere_requirement', 'age_requirement', 'responsibilities.1', 'sk

In [3]:
import pandas as pd

resume_df = pd.read_csv("UpdatedResumeDataSet.csv")

resume_df = resume_df.rename(columns={
    "Category": "category",
    "Resume": "resume_text"
})

resume_df["candidate_id"] = ["cand_" + str(i) for i in range(len(resume_df))]

resume_df["category"] = resume_df["category"].astype(str).str.strip()
resume_df["resume_text"] = resume_df["resume_text"].astype(str).str.strip()

resume_df = resume_df[["candidate_id", "category", "resume_text"]]
resume_df = resume_df.drop_duplicates()
resume_df = resume_df.dropna(subset=["resume_text"])

print(resume_df.shape)
print(resume_df.head(3))

(962, 3)
  candidate_id      category  \
0       cand_0  Data Science   
1       cand_1  Data Science   
2       cand_2  Data Science   

                                         resume_text  
0  Skills * Programming Languages: Python (pandas...  
1  Education Details \r\nMay 2013 to May 2017 B.E...  
2  Areas of Interest Deep Learning, Control Syste...  


In [4]:
resume_df.to_csv("resumes_clean.csv", index=False)

In [5]:
import re

def extract_github_url(text):
    if pd.isna(text):
        return None
    text = str(text)
    match = re.search(r'https?://github\.com/[A-Za-z0-9_.-]+', text, re.IGNORECASE)
    if match:
        return match.group(0)
    return None

resume_df["github_url"] = resume_df["resume_text"].apply(extract_github_url)

print(resume_df["github_url"].notna().sum())
print(resume_df[["candidate_id", "github_url"]].dropna().head(10))

0
Empty DataFrame
Columns: [candidate_id, github_url]
Index: []


In [6]:
def extract_github_username(url):
    if pd.isna(url) or url is None:
        return None
    url = str(url).strip().rstrip("/")
    return url.split("/")[-1]

resume_df["github_username"] = resume_df["github_url"].apply(extract_github_username)

print(resume_df[["github_url", "github_username"]].dropna().head(10))

Empty DataFrame
Columns: [github_url, github_username]
Index: []


In [7]:
resume_df.to_csv("resumes_with_github.csv", index=False)

In [8]:
try:
    import sentence_transformers
    from sentence_transformers import SentenceTransformer

    print("sentence-transformers is installed")
    print("Version:", sentence_transformers.__version__)

    model = SentenceTransformer("all-MiniLM-L6-v2")
    print("Model loaded successfully")

    test_texts = [
        "Built an AI hiring platform for resume screening and ranking",
        "Developed a recruitment system for parsing resumes and ranking applicants"
    ]

    embeddings = model.encode(test_texts)
    print("Number of embeddings:", len(embeddings))
    print("Embedding size:", len(embeddings[0]))

except Exception as e:
    print("Error:", e)

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sentence-transformers is installed
Version: 5.3.0


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12159.11it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully
Number of embeddings: 2
Embedding size: 384


In [9]:
import pandas as pd

resume_df = pd.read_csv("resumes_clean.csv")
resume_df = resume_df.fillna("")

sample_texts = resume_df["resume_text"].head(5).tolist()
embeddings = model.encode(sample_texts, convert_to_tensor=False)

print("Number of texts:", len(sample_texts))
print("Embedding shape of first resume:", len(embeddings[0]))

Number of texts: 5
Embedding shape of first resume: 384


In [10]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

text1 = "Built an AI hiring platform for resume screening and candidate ranking"
text2 = "Developed a recruitment system for parsing resumes and ranking applicants"
text3 = "Created a food delivery mobile application with restaurant listings"

vecs = model.encode([text1, text2, text3])

sim_12 = cosine_similarity([vecs[0]], [vecs[1]])[0][0]
sim_13 = cosine_similarity([vecs[0]], [vecs[2]])[0][0]

print("Similarity between text1 and text2:", sim_12)
print("Similarity between text1 and text3:", sim_13)

Similarity between text1 and text2: 0.63280857
Similarity between text1 and text3: 0.16188416


In [11]:
import pandas as pd
from sentence_transformers import SentenceTransformer

resume_df = pd.read_csv("resumes_clean.csv")
resume_df = resume_df.fillna("")

sample_df = resume_df.head(20).copy()

model = SentenceTransformer("all-MiniLM-L6-v2")

resume_embeddings = model.encode(
    sample_df["resume_text"].tolist(),
    convert_to_tensor=False
)

print("Rows embedded:", len(resume_embeddings))
print("Embedding size:", len(resume_embeddings[0]))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7875.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Rows embedded: 20
Embedding size: 384


In [12]:
job_text = """
We are looking for a Data Scientist with strong experience in Python,
machine learning, NLP, deep learning, SQL, analytics, and model building.
The candidate should have worked on predictive systems, data pipelines,
and AI-based applications.
"""

job_embedding = model.encode([job_text], convert_to_tensor=False)[0]

print("Job embedding size:", len(job_embedding))

Job embedding size: 384


In [13]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

resume_matrix = np.array(resume_embeddings)

scores = cosine_similarity(resume_matrix, [job_embedding]).flatten()

sample_df["semantic_score"] = scores

print(sample_df[["candidate_id", "category", "semantic_score"]].head())

  candidate_id      category  semantic_score
0       cand_0  Data Science        0.624061
1       cand_1  Data Science        0.383498
2       cand_2  Data Science        0.502110
3       cand_3  Data Science        0.357782
4       cand_4  Data Science        0.324492


In [14]:
ranked_df = sample_df.sort_values("semantic_score", ascending=False).reset_index(drop=True)

print(ranked_df[["candidate_id", "category", "semantic_score"]].head(10))

  candidate_id      category  semantic_score
0       cand_0  Data Science        0.624061
1      cand_10  Data Science        0.624061
2      cand_18  Data Science        0.557912
3       cand_8  Data Science        0.557912
4      cand_17  Data Science        0.526986
5       cand_7  Data Science        0.526986
6      cand_19  Data Science        0.507094
7       cand_9  Data Science        0.507094
8      cand_12  Data Science        0.502110
9       cand_2  Data Science        0.502110


In [15]:
for i in range(3):
    print("\n" + "="*100)
    print("Rank:", i + 1)
    print("Candidate ID:", ranked_df.loc[i, "candidate_id"])
    print("Category:", ranked_df.loc[i, "category"])
    print("Semantic Score:", ranked_df.loc[i, "semantic_score"])
    print("Resume Preview:")
    print(ranked_df.loc[i, "resume_text"][:1200])


Rank: 1
Candidate ID: cand_0
Category: Data Science
Semantic Score: 0.62406063
Resume Preview:
Skills * Programming Languages: Python (pandas, numpy, scipy, scikit-learn, matplotlib), Sql, Java, JavaScript/JQuery. * Machine learning: Regression, SVM, NaÃ¯ve Bayes, KNN, Random Forest, Decision Trees, Boosting techniques, Cluster Analysis, Word Embedding, Sentiment Analysis, Natural Language processing, Dimensionality reduction, Topic Modelling (LDA, NMF), PCA & Neural Nets. * Database Visualizations: Mysql, SqlServer, Cassandra, Hbase, ElasticSearch D3.js, DC.js, Plotly, kibana, matplotlib, ggplot, Tableau. * Others: Regular Expression, HTML, CSS, Angular 6, Logstash, Kafka, Python Flask, Git, Docker, computer vision - Open CV and understanding of Deep learning.Education Details 

Data Science Assurance Associate 

Data Science Assurance Associate - Ernst & Young LLP
Skill Details 
JAVASCRIPT- Exprience - 24 months
jQuery- Exprience - 24 months
Python- Exprience - 24 monthsCompany Deta

In [24]:
print(type(resume_df))
print(resume_df.columns.tolist())
print(resume_df.head(2))

<class 'pandas.DataFrame'>
['candidate_id', 'category', 'resume_text']
  candidate_id      category  \
0       cand_0  Data Science   
1       cand_1  Data Science   

                                         resume_text  
0  Skills * Programming Languages: Python (pandas...  
1  Education Details \r\nMay 2013 to May 2017 B.E...  


In [25]:
import pandas as pd

resume_df = pd.read_csv("UpdatedResumeDataSet.csv")

print("Before rename:", resume_df.columns.tolist())

resume_df = resume_df.rename(columns={
    "Category": "category",
    "Resume": "resume_text"
})

resume_df["candidate_id"] = ["cand_" + str(i) for i in range(len(resume_df))]
resume_df = resume_df[["candidate_id", "category", "resume_text"]].copy()
resume_df = resume_df.fillna("")

print("After rename:", resume_df.columns.tolist())
print(resume_df.head(2))

Before rename: ['Category', 'Resume']
After rename: ['candidate_id', 'category', 'resume_text']
  candidate_id      category  \
0       cand_0  Data Science   
1       cand_1  Data Science   

                                         resume_text  
0  Skills * Programming Languages: Python (pandas...  
1  Education Details \r\nMay 2013 to May 2017 B.E...  


In [26]:
print("category" in resume_df.columns)

True


In [28]:
import pandas as pd

resume_df = pd.read_csv("UpdatedResumeDataSet.csv")

print("Original columns:", resume_df.columns.tolist())

resume_df = resume_df.rename(columns={
    "Category": "category",
    "Resume": "resume_text"
})

resume_df["candidate_id"] = ["cand_" + str(i) for i in range(len(resume_df))]

resume_df = resume_df[["candidate_id", "category", "resume_text"]].copy()
resume_df = resume_df.fillna("")

print("Cleaned columns:", resume_df.columns.tolist())
print("Shape:", resume_df.shape)
print(resume_df.head(2))

Original columns: ['Category', 'Resume']
Cleaned columns: ['candidate_id', 'category', 'resume_text']
Shape: (962, 3)
  candidate_id      category  \
0       cand_0  Data Science   
1       cand_1  Data Science   

                                         resume_text  
0  Skills * Programming Languages: Python (pandas...  
1  Education Details \r\nMay 2013 to May 2017 B.E...  


In [31]:
mixed_df = (
    resume_df.groupby("category")
    .apply(lambda x: x.sample(min(5, len(x)), random_state=42))
    .reset_index(level=0)
    .reset_index(drop=True)
)

print("Mixed dataframe shape:", mixed_df.shape)
print("Mixed dataframe columns:", mixed_df.columns.tolist())
print("Category distribution in mixed_df:")
print(mixed_df["category"].value_counts())

Mixed dataframe shape: (125, 3)
Mixed dataframe columns: ['category', 'candidate_id', 'resume_text']
Category distribution in mixed_df:
category
Advocate                     5
Arts                         5
Automation Testing           5
Blockchain                   5
Business Analyst             5
Civil Engineer               5
Data Science                 5
Database                     5
DevOps Engineer              5
DotNet Developer             5
ETL Developer                5
Electrical Engineering       5
HR                           5
Hadoop                       5
Health and fitness           5
Java Developer               5
Mechanical Engineer          5
Network Security Engineer    5
Operations Manager           5
PMO                          5
Python Developer             5
SAP Developer                5
Sales                        5
Testing                      5
Web Designing                5
Name: count, dtype: int64


In [22]:
print("category" in resume_df.columns)

True


In [34]:
# Create balanced mixed sample from resume_df
mixed_df = (
    resume_df.groupby("category")
    .apply(lambda x: x.sample(min(5, len(x)), random_state=42))
    .reset_index(level=0)
    .reset_index(drop=True)
)

print("Mixed dataframe shape:", mixed_df.shape)
print("Mixed dataframe columns:", mixed_df.columns.tolist())
print("Category distribution in mixed_df:")
print(mixed_df["category"].value_counts())

Mixed dataframe shape: (125, 3)
Mixed dataframe columns: ['category', 'candidate_id', 'resume_text']
Category distribution in mixed_df:
category
Advocate                     5
Arts                         5
Automation Testing           5
Blockchain                   5
Business Analyst             5
Civil Engineer               5
Data Science                 5
Database                     5
DevOps Engineer              5
DotNet Developer             5
ETL Developer                5
Electrical Engineering       5
HR                           5
Hadoop                       5
Health and fitness           5
Java Developer               5
Mechanical Engineer          5
Network Security Engineer    5
Operations Manager           5
PMO                          5
Python Developer             5
SAP Developer                5
Sales                        5
Testing                      5
Web Designing                5
Name: count, dtype: int64


In [35]:
import pandas as pd
import numpy as np

# load raw resume dataset
resume_df = pd.read_csv("UpdatedResumeDataSet.csv")

print("Original shape:", resume_df.shape)
print("Original columns:", resume_df.columns.tolist())

# clean column names safely
resume_df.columns = [col.strip() for col in resume_df.columns]

# rename to standard schema
resume_df = resume_df.rename(columns={
    "Category": "category",
    "Resume": "resume_text"
})

# keep only required columns
resume_df = resume_df[["category", "resume_text"]].copy()

# remove missing values
resume_df = resume_df.dropna(subset=["category", "resume_text"]).reset_index(drop=True)

# make text clean
resume_df["category"] = resume_df["category"].astype(str).str.strip()
resume_df["resume_text"] = resume_df["resume_text"].astype(str).str.strip()

# create candidate id
resume_df.insert(0, "candidate_id", ["cand_" + str(i) for i in range(len(resume_df))])

print("\nCleaned shape:", resume_df.shape)
print("Cleaned columns:", resume_df.columns.tolist())
print("\nSample rows:")
print(resume_df.head())

# save clean version
resume_df.to_csv("resumes_clean.csv", index=False)
print("\nSaved: resumes_clean.csv")

Original shape: (962, 2)
Original columns: ['Category', 'Resume']

Cleaned shape: (962, 3)
Cleaned columns: ['candidate_id', 'category', 'resume_text']

Sample rows:
  candidate_id      category  \
0       cand_0  Data Science   
1       cand_1  Data Science   
2       cand_2  Data Science   
3       cand_3  Data Science   
4       cand_4  Data Science   

                                         resume_text  
0  Skills * Programming Languages: Python (pandas...  
1  Education Details \r\nMay 2013 to May 2017 B.E...  
2  Areas of Interest Deep Learning, Control Syste...  
3  Skills â¢ R â¢ Python â¢ SAP HANA â¢ Table...  
4  Education Details \r\n MCA   YMCAUST,  Faridab...  

Saved: resumes_clean.csv


In [36]:
mixed_df = (
    resume_df.groupby("category", group_keys=False)
    .apply(lambda x: x.sample(min(5, len(x)), random_state=42))
    .reset_index(drop=True)
)

In [37]:
resume_df.head(20)

,candidate_id,category,resume_text
0,cand_0,Data Science,Skills * Programming Languages: Python (pandas...
1,cand_1,Data Science,Education Details \r\nMay 2013 to May 2017 B.E...
2,cand_2,Data Science,"Areas of Interest Deep Learning, Control Syste..."
3,cand_3,Data Science,Skills â¢ R â¢ Python â¢ SAP HANA â¢ Table...
4,cand_4,Data Science,"Education Details \r\n MCA YMCAUST, Faridab..."
5,cand_5,Data Science,"SKILLS C Basics, IOT, Python, MATLAB, Data Sci..."
6,cand_6,Data Science,Skills â¢ Python â¢ Tableau â¢ Data Visuali...
7,cand_7,Data Science,Education Details \r\n B.Tech Rayat and Bahr...
8,cand_8,Data Science,Personal Skills â¢ Ability to quickly grasp t...
9,cand_9,Data Science,Expertise â Data and Quantitative Analysis â...


In [39]:
mixed_df = (
    resume_df.groupby("category")
    .apply(lambda x: x.sample(min(5, len(x)), random_state=42))
    .reset_index(level=0)
    .reset_index(drop=True)
)

print("Mixed shape:", mixed_df.shape)
print("\nCategory counts:")
print(mixed_df["category"].value_counts())

mixed_df.head()

Mixed shape: (125, 3)

Category counts:
category
Advocate                     5
Arts                         5
Automation Testing           5
Blockchain                   5
Business Analyst             5
Civil Engineer               5
Data Science                 5
Database                     5
DevOps Engineer              5
DotNet Developer             5
ETL Developer                5
Electrical Engineering       5
HR                           5
Hadoop                       5
Health and fitness           5
Java Developer               5
Mechanical Engineer          5
Network Security Engineer    5
Operations Manager           5
PMO                          5
Python Developer             5
SAP Developer                5
Sales                        5
Testing                      5
Web Designing                5
Name: count, dtype: int64


,category,candidate_id,resume_text
0,Advocate,cand_84,"TECHNICAL QUALIFICATIONS: â¢ Windows, Ms. Off..."
1,Advocate,cand_101,Skills Legal Writing Efficient researcher Lega...
2,Advocate,cand_99,QUALIFICATION: Introduction to Computer EXTRAE...
3,Advocate,cand_85,"Education Details \r\n B.Com, LL.B., Univers..."
4,Advocate,cand_92,Good grasping quality and skillful work Educat...


In [40]:
from sentence_transformers import SentenceTransformer, util

# load model
model = SentenceTransformer("all-MiniLM-L6-v2")

# sample job description
job_text = """
We are looking for a Data Scientist with experience in Python, machine learning,
NLP, SQL, data analysis, feature engineering, model building, and predictive analytics.
Experience with deep learning, statistics, and deploying ML models is a plus.
"""

# create embeddings
resume_embeddings = model.encode(
    mixed_df["resume_text"].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True
)

job_embedding = model.encode(job_text, convert_to_tensor=True)

# cosine similarity
scores = util.cos_sim(job_embedding, resume_embeddings)[0].cpu().numpy()

# add score to dataframe
mixed_df["semantic_score"] = scores

# rank candidates
mixed_ranked_df = mixed_df.sort_values(by="semantic_score", ascending=False).reset_index(drop=True)

print(mixed_ranked_df[["candidate_id", "category", "semantic_score"]].head(10))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5699.76it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 4/4 [00:01<00:00,  2.58it/s]


  candidate_id          category  semantic_score
0      cand_26      Data Science        0.520857
1      cand_16      Data Science        0.520857
2      cand_19      Data Science        0.497813
3     cand_575  Python Developer        0.440410
4     cand_768            Hadoop        0.429538
5      cand_15      Data Science        0.420127
6     cand_577  Python Developer        0.400742
7       cand_4      Data Science        0.360932
8     cand_447     SAP Developer        0.360436
9     cand_604   DevOps Engineer        0.358692


In [41]:
mixed_df.to_csv("mixed_resumes_sample.csv", index=False)
mixed_ranked_df.to_csv("mixed_ranked_resumes.csv", index=False)

print("Saved:")
print("- mixed_resumes_sample.csv")
print("- mixed_ranked_resumes.csv")

Saved:
- mixed_resumes_sample.csv
- mixed_ranked_resumes.csv


In [42]:
import re

def extract_github_url(text):
    if pd.isna(text):
        return None
    
    text = str(text)
    
    patterns = [
        r'https?://github\.com/[A-Za-z0-9_.-]+',
        r'www\.github\.com/[A-Za-z0-9_.-]+',
        r'github\.com/[A-Za-z0-9_.-]+'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            url = match.group(0)
            if not url.startswith("http"):
                url = "https://" + url
            return url
    
    return None

resume_df["github_url"] = resume_df["resume_text"].apply(extract_github_url)

print(resume_df[["candidate_id", "category", "github_url"]].head(20))
print("\nGitHub URLs found:", resume_df["github_url"].notna().sum())

resume_df.to_csv("resumes_with_github_urls.csv", index=False)

   candidate_id      category                      github_url
0        cand_0  Data Science                             NaN
1        cand_1  Data Science  https://github.com/rathorology
2        cand_2  Data Science                             NaN
3        cand_3  Data Science                             NaN
4        cand_4  Data Science                             NaN
5        cand_5  Data Science                             NaN
6        cand_6  Data Science                             NaN
7        cand_7  Data Science                             NaN
8        cand_8  Data Science                             NaN
9        cand_9  Data Science                             NaN
10      cand_10  Data Science                             NaN
11      cand_11  Data Science  https://github.com/rathorology
12      cand_12  Data Science                             NaN
13      cand_13  Data Science                             NaN
14      cand_14  Data Science                             NaN
15      

In [43]:
def extract_github_username(url):
    if pd.isna(url) or url is None:
        return None
    
    match = re.search(r'github\.com/([A-Za-z0-9_.-]+)', url, flags=re.IGNORECASE)
    if match:
        return match.group(1)
    return None

resume_df["github_username"] = resume_df["github_url"].apply(extract_github_username)

print(resume_df[["candidate_id", "github_url", "github_username"]].head(20))

resume_df.to_csv("resumes_with_github_usernames.csv", index=False)

   candidate_id                      github_url github_username
0        cand_0                             NaN             NaN
1        cand_1  https://github.com/rathorology     rathorology
2        cand_2                             NaN             NaN
3        cand_3                             NaN             NaN
4        cand_4                             NaN             NaN
5        cand_5                             NaN             NaN
6        cand_6                             NaN             NaN
7        cand_7                             NaN             NaN
8        cand_8                             NaN             NaN
9        cand_9                             NaN             NaN
10      cand_10                             NaN             NaN
11      cand_11  https://github.com/rathorology     rathorology
12      cand_12                             NaN             NaN
13      cand_13                             NaN             NaN
14      cand_14                         

In [44]:
import requests
import pandas as pd
import numpy as np
import re
import time
from sentence_transformers import SentenceTransformer, util

In [ ]:
GITHUB_TOKEN = None  # optional: put token here if you have one GitHub API helper functions This fetches public profile + repos.



headers = {}
if GITHUB_TOKEN:
    headers["Authorization"] = f"token {GITHUB_TOKEN}"

def fetch_github_profile(username):
    if not username:
        return None
    
    url = f"https://api.github.com/users/{username}"
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        return response.json()
    else:
        return None

def fetch_github_repos(username):
    if not username:
        return []
    
    url = f"https://api.github.com/users/{username}/repos?per_page=100&sort=updated"
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        return response.json()
    else:
        return []

In [51]:
#test on one username first Before running it on the whole dataframe, test one row.


sample_users = resume_df["github_username"].dropna().unique().tolist()[:5]
print(sample_users)

if sample_users:
    test_user = sample_users[0]
    print("Testing username:", test_user)

    profile = fetch_github_profile(test_user)
    repos = fetch_github_repos(test_user)

    print("\nProfile:")
    print(profile)

    print("\nNumber of repos:", len(repos))
    if len(repos) > 0:
        print("\nFirst repo keys:", repos[0].keys())
        print("\nFirst repo sample:", repos[0])

['rathorology']
Testing username: rathorology

Profile:
{'login': 'rathorology', 'id': 32793233, 'node_id': 'MDQ6VXNlcjMyNzkzMjMz', 'avatar_url': 'https://avatars.githubusercontent.com/u/32793233?v=4', 'gravatar_id': '', 'url': 'https://api.github.com/users/rathorology', 'html_url': 'https://github.com/rathorology', 'followers_url': 'https://api.github.com/users/rathorology/followers', 'following_url': 'https://api.github.com/users/rathorology/following{/other_user}', 'gists_url': 'https://api.github.com/users/rathorology/gists{/gist_id}', 'starred_url': 'https://api.github.com/users/rathorology/starred{/owner}{/repo}', 'subscriptions_url': 'https://api.github.com/users/rathorology/subscriptions', 'organizations_url': 'https://api.github.com/users/rathorology/orgs', 'repos_url': 'https://api.github.com/users/rathorology/repos', 'events_url': 'https://api.github.com/users/rathorology/events{/privacy}', 'received_events_url': 'https://api.github.com/users/rathorology/received_events', 't

In [ ]:
#Step 11 — convert repos into one text block per user that we can embed and compare with job description.
def build_repo_text(repos):
    repo_texts = []
    
    for repo in repos:
        name = str(repo.get("name", ""))
        description = str(repo.get("description", ""))
        language = str(repo.get("language", ""))
        topics = repo.get("topics", [])
        if topics is None:
            topics = []
        topics_text = " ".join([str(t) for t in topics])
        
        text = f"""
        Repo Name: {name}
        Description: {description}
        Language: {language}
        Topics: {topics_text}
        """
        repo_texts.append(text.strip())
    
    return "\n\n".join(repo_texts)

In [52]:
if sample_users:
    repos = fetch_github_repos(sample_users[0])
    repo_text = build_repo_text(repos)
    print(repo_text[:2000])  # preview

Repo Name: Image-table-to-text-
        Description: Extracting tabular data from the image and storing it in CSV.
        Language: Python
        Topics: kraken ocr-python opencv

Repo Name: forecasting_and_tuning
        Description: Time series forecasting with hyperparameter tuning
        Language: Jupyter Notebook
        Topics:

Repo Name: yolo_medical
        Description: Yolov8 on medical imaging
        Language: Python
        Topics:

Repo Name: Loan-Defaulters
        Description: None
        Language: Python
        Topics:

Repo Name: TT_table_detection
        Description: Detecting table tennis table
        Language: Python
        Topics: opencv

Repo Name: PASTIS_Panoptic
        Description: Setup of utae-paps repository with EDA
        Language: Jupyter Notebook
        Topics:

Repo Name: Segmentation-DeepLab-v3-
        Description: Image Segmentation using DeepLab v3+ and Tensorflow 2.0
        Language: Python
        Topics:

Repo Name: SSD-Multi-face-mas

In [53]:
#Step 12 — fetch GitHub evidence for all candidates
github_repo_counts = []
github_repo_texts = []
github_profiles = []

for username in resume_df["github_username"]:
    if pd.isna(username) or username is None:
        github_profiles.append(None)
        github_repo_counts.append(0)
        github_repo_texts.append("")
        continue
    
    try:
        profile = fetch_github_profile(username)
        repos = fetch_github_repos(username)
        repo_text = build_repo_text(repos)
        
        github_profiles.append(profile)
        github_repo_counts.append(len(repos))
        github_repo_texts.append(repo_text)
        
        time.sleep(0.5)  # avoid hitting rate limits
    except Exception as e:
        print(f"Error for {username}: {e}")
        github_profiles.append(None)
        github_repo_counts.append(0)
        github_repo_texts.append("")

resume_df["github_repo_count"] = github_repo_counts
resume_df["github_repo_text"] = github_repo_texts

print(resume_df[["candidate_id", "github_username", "github_repo_count"]].head(20))

   candidate_id github_username  github_repo_count
0        cand_0             NaN                  0
1        cand_1     rathorology                 26
2        cand_2             NaN                  0
3        cand_3             NaN                  0
4        cand_4             NaN                  0
5        cand_5             NaN                  0
6        cand_6             NaN                  0
7        cand_7             NaN                  0
8        cand_8             NaN                  0
9        cand_9             NaN                  0
10      cand_10             NaN                  0
11      cand_11     rathorology                 26
12      cand_12             NaN                  0
13      cand_13             NaN                  0
14      cand_14             NaN                  0
15      cand_15             NaN                  0
16      cand_16             NaN                  0
17      cand_17             NaN                  0
18      cand_18             NaN

In [54]:
resume_df.to_csv("resumes_with_github_repo_text.csv", index=False)
print("Saved: resumes_with_github_repo_text.csv")

Saved: resumes_with_github_repo_text.csv


In [56]:
#Step 13 — semantic match between resume and GitHub evidence
model = SentenceTransformer("all-MiniLM-L6-v2")

def compute_resume_github_similarity(resume_text, github_text):
    if not resume_text or not github_text:
        return 0.0
    
    try:
        emb1 = model.encode(str(resume_text), convert_to_tensor=True)
        emb2 = model.encode(str(github_text), convert_to_tensor=True)
        score = util.cos_sim(emb1, emb2).item()
        return float(score)
    except Exception as e:
        print("Similarity error:", e)
        return 0.0

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6159.56it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [57]:
#apply it 
resume_df["project_match_score"] = resume_df.apply(
    lambda row: compute_resume_github_similarity(
        row["resume_text"], row["github_repo_text"]
    ),
    axis=1
)

print(
    resume_df[[
        "candidate_id",
        "github_username",
        "github_repo_count",
        "project_match_score"
    ]].sort_values(by="project_match_score", ascending=False).head(10)
)

    candidate_id github_username  github_repo_count  project_match_score
31       cand_31     rathorology                 26             0.397143
1         cand_1     rathorology                 26             0.397143
21       cand_21     rathorology                 26             0.397143
11       cand_11     rathorology                 26             0.397143
640     cand_640             NaN                  0             0.000000
644     cand_644             NaN                  0             0.000000
643     cand_643             NaN                  0             0.000000
642     cand_642             NaN                  0             0.000000
641     cand_641             NaN                  0             0.000000
639     cand_639             NaN                  0             0.000000


In [58]:
resume_df.to_csv("resumes_with_project_match_score.csv", index=False)
print("Saved: resumes_with_project_match_score.csv")

Saved: resumes_with_project_match_score.csv


In [59]:
#Step 14 — add activity score
def normalize_repo_count(count, max_count):
    if max_count == 0:
        return 0.0
    return count / max_count

max_repo_count = resume_df["github_repo_count"].max()

resume_df["github_activity_score"] = resume_df["github_repo_count"].apply(
    lambda x: normalize_repo_count(x, max_repo_count)
)

print(
    resume_df[[
        "candidate_id",
        "github_username",
        "github_repo_count",
        "github_activity_score"
    ]].sort_values(by="github_activity_score", ascending=False).head(10)
)

    candidate_id github_username  github_repo_count  github_activity_score
31       cand_31     rathorology                 26                    1.0
1         cand_1     rathorology                 26                    1.0
21       cand_21     rathorology                 26                    1.0
11       cand_11     rathorology                 26                    1.0
640     cand_640             NaN                  0                    0.0
644     cand_644             NaN                  0                    0.0
643     cand_643             NaN                  0                    0.0
642     cand_642             NaN                  0                    0.0
641     cand_641             NaN                  0                    0.0
639     cand_639             NaN                  0                    0.0


In [61]:
merged_df = resume_df.merge(
    mixed_ranked_df[["candidate_id", "semantic_score"]],
    on="candidate_id",
    how="left"
)

merged_df["semantic_score"] = merged_df["semantic_score"].fillna(0)

merged_df["final_score"] = (
    0.60 * merged_df["semantic_score"] +
    0.25 * merged_df["project_match_score"] +
    0.15 * merged_df["github_activity_score"]
)

final_ranked_df = merged_df.sort_values(by="final_score", ascending=False).reset_index(drop=True)

print(
    final_ranked_df[[
        "candidate_id",
        "category",
        "semantic_score",
        "project_match_score",
        "github_activity_score",
        "final_score"
    ]].head(20)
)

   candidate_id            category  semantic_score  project_match_score  \
0       cand_16        Data Science        0.520857             0.000000   
1       cand_26        Data Science        0.520857             0.000000   
2       cand_19        Data Science        0.497813             0.000000   
3      cand_575    Python Developer        0.440410             0.000000   
4      cand_768              Hadoop        0.429538             0.000000   
5       cand_15        Data Science        0.420127             0.000000   
6       cand_21        Data Science        0.000000             0.397143   
7        cand_1        Data Science        0.000000             0.397143   
8       cand_31        Data Science        0.000000             0.397143   
9       cand_11        Data Science        0.000000             0.397143   
10     cand_577    Python Developer        0.400742             0.000000   
11       cand_4        Data Science        0.360932             0.000000   
12     cand_